In [ ]:
FILES = {'tta_px1.py': '"""\nPX1 stage 5: label-free adaptation to each unseen flood, combined with the miss-rate guarantee.\n\nFor every held-out flood E and variant (V0 radar only, V1 radar + elevation) we take the\nleave-one-event-out model that never saw E and adapt it to E\'s chips WITHOUT labels:\n\n  none   no adaptation (reproduces conformal_px1.py)\n  tent   TENT: minimise prediction entropy, update batch-norm parameters only\n  st     mean-teacher self-training (the recipe of Chen et al. 2026): EMA teacher makes\n         confident pseudo-labels (p > 0.9 water, p < 0.1 dry) on a weak view, the student\n         learns them on a strong view (speckle-like noise, +-1.5 dB radiometric shift)\n  phys   st + physics-checked pseudo-labels (ours). Rules fixed in advance (same radar\n         thresholds as the label audit):\n           water pseudo-label dropped if the pixel is radar-bright (VV > -15 dB) or\n           physically implausible for ponding (> 15 m above the chip\'s low ground, or slope > 8 deg)\n           a pixel that is radar-dark (VV < -18 and VH < -25 dB), low (< 5 m) and flat (< 3 deg)\n           becomes a water pseudo-label unless the teacher is confident it is dry (then ignored)\n         Elevation is used here only to check pseudo-labels, so it also helps the radar-only model.\n\nLabels of E are used only for scoring. Every setting is fixed before seeing any result\n(200 steps, batch 8, 256 crops, Adam; TENT lr 1e-3, self-training lr 1e-4, EMA 0.99).\n\nThen, as in conformal_px1.py, the threshold for E is calibrated on the OTHER floods, each\nadapted the same way by its own held-out model, and scored on E.\nA classical physics baseline (Otsu threshold on VV per flood, no model) is reported too.\n\nAblations (physics check with one rule only): --methods phys_drop,phys_add\n\nUsage:  python tta_px1.py --data data --runs runs_loeo     (resumes from runs_loeo/tta_hist.npz)\n        python tta_px1.py --analyse-only\n"""\nimport argparse\nimport copy\nimport json\nimport os\nimport time\n\nos.environ.setdefault("GDAL_CACHEMAX", "64")\n\nimport numpy as np\n\nfrom conformal_px1 import ALPHAS, BINS, EVENTS, VARIANTS, curves\n\nMETHODS = ["none", "tent", "st", "phys"]\nABLATIONS = ["phys_drop", "phys_add"]  # physics check with only the drop rule / only the add rule\nCOMBOS = ["adabn", "adabn_phys", "tent_phys"]  # added after the first Kuro Siwo results: recalibrate, then physics-checked self-training\nADAPTIVE = ["physa", "adabn_physa"]  # per-flood radar thresholds + HAND gating (add rule only)\nALL_METHODS = METHODS + ABLATIONS + COMBOS + ADAPTIVE\nALL_METHODS += ["tent_batch"]  # separate key: never reuse historical tent histograms\nSTEPS, BATCH, CROP = 200, 8, 256\nLR_TENT, LR_ST, EMA, P_HI, P_LO = 1e-3, 1e-4, 0.99, 0.9, 0.1\nADABN_BATCHES = 50\nVV_BRIGHT, VV_DARK, VH_DARK = -15.0, -18.0, -25.0\nRELH_HIGH, SLOPE_HIGH, RELH_LOW, SLOPE_LOW = 15.0, 8.0, 5.0, 3.0\n# adaptive rule (fixed before any run): per flood, Otsu split of all VV (and VH) pixels; "dark" = below the\n# midpoint between the dark-mode mean and the Otsu threshold, capped at -15 dB (VV) / -22 dB (VH) so that a\n# flood with little water cannot turn dry land into water; low = HAND < 5 m; flat = slope < 3 deg\nHAND_LOW, VV_CAP, VH_CAP = 5.0, -15.0, -22.0\n\n\ndef otsu_dark(values):\n    """Otsu threshold of a 1-D sample and the mean of the class below it."""\n    v = values[values > -49.9]\n    hist, edges = np.histogram(v, bins=256)\n    c = (edges[:-1] + edges[1:]) / 2\n    w0 = np.cumsum(hist); w1 = w0[-1] - w0\n    s0 = np.cumsum(hist * c)\n    mu0 = s0 / np.maximum(w0, 1); mu1 = (s0[-1] - s0) / np.maximum(w1, 1)\n    k = int(np.argmax(w0 * w1 * (mu0 - mu1) ** 2))\n    return float(c[k]), float(mu0[k])\n\n\ndef adaptive_dark(raw_list):\n    """Per-flood dark thresholds (VV, VH) from the flood\'s own backscatter, no labels."""\n    out = []\n    for ch, cap in [(0, VV_CAP), (1, VH_CAP)]:\n        v = np.concatenate([r[ch].ravel()[::7] for r in raw_list])\n        t, mu = otsu_dark(v)\n        out.append(min(cap, (t + mu) / 2))\n    return out\n\n\ndef hand_channel(hand, terr):\n    """Third terrain channel: HAND / 50 m (falls back to the chip\'s relative height if HAND is missing)."""\n    h = terr[0:1] if hand is None else np.clip(hand[None].astype(np.float32), 0, 100) / 50.0\n    return np.concatenate([terr, h], 0).astype(np.float32)\n\n\ndef load_event(args, e, every, clean):\n    from sen1floods11_unet_baseline import event_of, load_label, load_s1\n    from train_px1 import load_dem, terrain_features\n    ev = {"raw": [], "terr": [], "lab": [], "clean": []}\n    for s, l in [p for p in every if event_of(p[0]) == e]:\n        s1_path = os.path.join(args.data, "S1Hand", s)\n        ev["raw"].append(load_s1(s1_path).astype(np.float32))\n        hp = os.path.join(args.data, "HAND", s.replace("S1Hand", "HAND"))\n        hand = None\n        if os.path.exists(hp):\n            import rasterio\n            with rasterio.open(hp) as src:\n                hand = src.read(1)\n        ev["hand_ok"] = ev.get("hand_ok", 0) + (hand is not None)\n        ev["terr"].append(hand_channel(hand, terrain_features(load_dem(args.data, s, s1_path), s1_path)))\n        ev["lab"].append(load_label(os.path.join(args.data, "LabelHand", l)))\n        ev["clean"].append(s in clean)\n    return ev\n\n\ndef run_one(method, v, e, ev, mean, std, args, device, ckpt=None):\n    import torch\n    import torch.nn.functional as F\n    from sen1floods11_unet_baseline import IGNORE, UNet\n\n    use_dem = VARIANTS[v] == "s1dem"\n    torch.manual_seed(args.seed)\n    gen = np.random.default_rng(args.seed)\n    m_t = torch.tensor(mean, device=device)[None, :, None, None]\n    s_t = torch.tensor(std, device=device)[None, :, None, None]\n    raw = [torch.from_numpy(r).to(device) for r in ev["raw"]]\n    terr = [torch.from_numpy(t).to(device) for t in ev["terr"]]\n\n    def inp(r, t):\n        x = (r - m_t) / s_t\n        return torch.cat([x, t[:, :2]], 1) if use_dem else x\n\n    model = UNet(cin=4 if use_dem else 2).to(device)\n    ckpt = ckpt or os.path.join(args.runs, f"{v}_hold{e}_seed{args.model_seed}", "model.pt")\n    model.load_state_dict(torch.load(ckpt, map_location=device))\n\n    def crops():\n        rs, ts = [], []\n        for i in gen.integers(0, len(raw), BATCH):\n            H, W = raw[i].shape[1:]\n            c = min(CROP, H, W)\n            r0, c0 = gen.integers(0, H - c + 1), gen.integers(0, W - c + 1)\n            r, t = raw[i][:, r0:r0 + c, c0:c0 + c], terr[i][:, r0:r0 + c, c0:c0 + c]\n            k = int(gen.integers(0, 4))\n            r, t = torch.rot90(r, k, (1, 2)), torch.rot90(t, k, (1, 2))\n            if gen.random() < 0.5:\n                r, t = r.flip(2), t.flip(2)\n            rs.append(r); ts.append(t)\n        return torch.stack(rs), torch.stack(ts)\n\n    # optional first stage: re-estimate batch-norm statistics (adabn) or TENT, then optional self-training\n    pre, base = None, method\n    if method in ("adabn", "tent", "tent_batch"):\n        pre, base = method, "none"\n    elif method.startswith(("adabn_", "tent_")):\n        pre, base = method.split("_", 1)\n    if pre == "adabn":\n        bns = [mod for mod in model.modules() if isinstance(mod, torch.nn.BatchNorm2d)]\n        for b in bns:\n            b.reset_running_stats(); b.momentum = None  # cumulative average over the target batches\n        model.train()\n        with torch.no_grad():\n            for _ in range(ADABN_BATCHES):\n                r, t = crops()\n                model(inp(r, t))\n        for b in bns:\n            b.momentum = 0.1\n    if pre in ("tent", "tent_batch"):\n        model.train()\n        if pre == "tent_batch":\n            # Official TENT BN configuration, also retained during final.eval().\n            # Event-wise 200-step adaptation and pixel-wise entropy remain PX1 choices.\n            for mod in model.modules():\n                if isinstance(mod, torch.nn.BatchNorm2d):\n                    mod.track_running_stats = False\n                    mod.running_mean = None\n                    mod.running_var = None\n        params = [p for mod in model.modules() if isinstance(mod, torch.nn.BatchNorm2d) for p in (mod.weight, mod.bias)]\n        for p in model.parameters():\n            p.requires_grad_(False)\n        for p in params:\n            p.requires_grad_(True)\n        opt = torch.optim.Adam(params, lr=LR_TENT)\n        for _ in range(args.steps):\n            r, t = crops()\n            p = model(inp(r, t)).softmax(1)\n            loss = -(p * torch.log(p.clamp_min(1e-6))).sum(1).mean()\n            opt.zero_grad(); loss.backward(); opt.step()\n        for p in model.parameters():\n            p.requires_grad_(True)\n    final = model\n    if base in ("st", "phys", "phys_drop", "phys_add", "physa"):\n        method = base\n        if method == "physa":\n            tv, th = adaptive_dark(ev["raw"])\n            print(f"    adaptive dark thresholds VV {tv:.1f} dB, VH {th:.1f} dB; HAND for {ev.get(\'hand_ok\', 0)}/{len(raw)} chips", flush=True)\n        teacher = copy.deepcopy(model).eval()\n        for p in teacher.parameters():\n            p.requires_grad_(False)\n        model.train()\n        opt = torch.optim.Adam(model.parameters(), lr=LR_ST)\n        ce = torch.nn.CrossEntropyLoss(weight=torch.tensor([1.0, 3.0], device=device), ignore_index=IGNORE)\n        kept = []\n        for _ in range(args.steps):\n            r, t = crops()\n            with torch.no_grad():\n                pt = teacher(inp(r, t)).softmax(1)[:, 1]\n                y = torch.full_like(pt, IGNORE, dtype=torch.long)\n                y[pt > P_HI] = 1\n                y[pt < P_LO] = 0\n                relh, slope = t[:, 0] * 50, t[:, 1] * 10\n                if method in ("phys", "phys_drop"):\n                    implausible = (r[:, 0] > VV_BRIGHT) | (relh > RELH_HIGH) | (slope > SLOPE_HIGH)\n                    y[(y == 1) & implausible] = IGNORE\n                if method == "physa":\n                    hand = t[:, 2] * 50\n                    dark_low = (r[:, 0] < tv) & (r[:, 1] < th) & (hand < HAND_LOW) & (slope < SLOPE_LOW)\n                    y[dark_low & (pt >= P_LO)] = 1\n                    y[dark_low & (pt < P_LO)] = IGNORE\n                if method in ("phys", "phys_add"):\n                    dark_low = (r[:, 0] < VV_DARK) & (r[:, 1] < VH_DARK) & (relh < RELH_LOW) & (slope < SLOPE_LOW)\n                    y[dark_low & (pt >= P_LO)] = 1\n                    y[dark_low & (pt < P_LO)] = IGNORE\n                kept.append(float((y != IGNORE).float().mean()))\n                off = (torch.rand(r.shape[0], 1, 1, 1, device=device) * 3 - 1.5)\n            xs = inp(r + off, t)\n            xs = torch.cat([xs[:, :2] + 0.1 * torch.randn_like(xs[:, :2]), xs[:, 2:]], 1)\n            loss = ce(model(xs), y)\n            opt.zero_grad(); loss.backward(); opt.step()\n            with torch.no_grad():\n                for pt_, ps_ in zip(teacher.parameters(), model.parameters()):\n                    pt_.mul_(EMA).add_(ps_.detach(), alpha=1 - EMA)\n                for bt, bs in zip(teacher.buffers(), model.buffers()):\n                    if bt.dtype.is_floating_point:\n                        bt.mul_(EMA).add_(bs, alpha=1 - EMA)\n                    else:\n                        bt.copy_(bs)\n        print(f"    pseudo-labelled share of pixels {np.mean(kept):.3f}", flush=True)\n        final = teacher\n\n    final.eval()\n    h = np.zeros((2, 2, BINS), np.int64)\n    with torch.no_grad():\n        for i in range(len(raw)):\n            p = final(inp(raw[i][None], terr[i][None])).softmax(1)[0, 1]\n            b = (p * BINS).long().clamp(0, BINS - 1)\n            lab = torch.from_numpy(ev["lab"][i].astype(np.int64)).to(device)\n            for sub in ([0, 1] if ev["clean"][i] else [0]):\n                for c in (0, 1):\n                    h[sub, c] += torch.bincount(b[lab == c], minlength=BINS).cpu().numpy()\n    return h\n\n\ndef otsu_counts(ev):\n    """Classical physics baseline: one Otsu threshold on VV per flood; water = VV below it."""\n    vv = np.concatenate([r[0].ravel() for r in ev["raw"]])\n    hist, edges = np.histogram(vv[vv > -49.9], bins=256)\n    c = (edges[:-1] + edges[1:]) / 2\n    w0 = np.cumsum(hist); w1 = w0[-1] - w0\n    mu0 = np.cumsum(hist * c) / np.maximum(w0, 1)\n    mu1 = (np.sum(hist * c) - np.cumsum(hist * c)) / np.maximum(w1, 1)\n    thr = c[np.argmax(w0 * w1 * (mu0 - mu1) ** 2)]\n    out = np.zeros((2, 3), np.int64)  # [all/clean][tp, fp, fn]\n    for r, lab, cl in zip(ev["raw"], ev["lab"], ev["clean"]):\n        pred = r[0] < thr\n        cnt = [np.sum(pred & (lab == 1)), np.sum(pred & (lab == 0)), np.sum(~pred & (lab == 1))]\n        for sub in ([0, 1] if cl else [0]):\n            out[sub] += cnt\n    return out, float(thr)\n\n\ndef save(path, d):\n    tmp = path[:-4] + ".tmp.npz"\n    np.savez_compressed(tmp, **d)\n    os.replace(tmp, path)\n\n\ndef phase1(args):\n    import torch\n    from sen1floods11_unet_baseline import channel_stats, event_of, read_split\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    path = os.path.join(args.runs, f"tta_hist{tag(args)}.npz")\n    done = dict(np.load(path)) if os.path.exists(path) else {}\n    with open(os.path.join("audit", "clean_chips.txt")) as f:\n        clean = {ln.strip() for ln in f if ln.strip()}\n    splits = {k: read_split(args.data, k) for k in ["train", "valid", "test", "bolivia"]}\n    every = splits["train"] + splits["valid"] + splits["test"] + splits["bolivia"]\n    for e in EVENTS:\n        todo = [(m, v) for v in VARIANTS for m in args.methods\n                if f"{m}|{v}|{e}" not in done and os.path.exists(os.path.join(args.runs, f"{v}_hold{e}_seed{args.model_seed}", "model.pt"))]\n        if not todo and f"otsu|{e}" in done:\n            continue\n        ev = load_event(args, e, every, clean)\n        if not ev["raw"]:\n            continue\n        mean, std = channel_stats(args.data, [p for p in splits["train"] if event_of(p[0]) != e])\n        if f"otsu|{e}" not in done:\n            done[f"otsu|{e}"], thr = otsu_counts(ev)\n            print(f"{e}: Otsu VV threshold {thr:.1f} dB", flush=True)\n        for m, v in todo:\n            t0 = time.time()\n            h = run_one(m, v, e, ev, mean, std, args, device)\n            done[f"{m}|{v}|{e}"] = h\n            fnr, _, iou = curves(h[0])\n            print(f"{m:9s} {v} hold {e:9s}: miss rate at 0.5 {fnr[BINS // 2]:.3f}, IoU {iou[BINS // 2]:.3f}, "\n                  f"{time.time() - t0:.0f}s", flush=True)\n            save(path, done)\n        save(path, done)\n        del ev\n        if device == "cuda":\n            torch.cuda.empty_cache()\n\n\ndef tag(args):\n    return "" if args.seed == 0 and args.model_seed == 0 else f"_m{args.model_seed}_a{args.seed}"\n\n\ndef compute(data, events=None):\n    events = events or EVENTS\n    out = {}\n    k05 = BINS // 2\n    for v in VARIANTS:\n        for sub, sub_name in [(0, "all"), (1, "clean")]:\n            block = {}\n            for m in [m for m in ALL_METHODS if any(k.startswith(m + "|") for k in data)]:\n                cur = {e: curves(data[f"{m}|{v}|{e}"][sub]) for e in events\n                       if f"{m}|{v}|{e}" in data and data[f"{m}|{v}|{e}"][sub, 1].sum() > 0}\n                if len(cur) < 2:\n                    continue\n                per = {}\n                for e in cur:\n                    others = np.mean([cur[o][0] for o in cur if o != e], axis=0)\n                    n = len(cur) - 1\n                    fnr, prec, iou = cur[e]\n                    per[e] = {"default": {"fnr": fnr[k05], "iou": iou[k05], "precision": prec[k05]}}\n                    for a in ALPHAS:\n                        ok = np.where(n / (n + 1) * others + 1 / (n + 1) <= a)[0]\n                        k = int(ok.max()) if len(ok) else 0\n                        per[e][f"crc_{a}"] = {"threshold": k / BINS, "fnr": fnr[k], "iou": iou[k], "precision": prec[k]}\n                summ = {}\n                for rule in ["default"] + [f"crc_{a}" for a in ALPHAS]:\n                    f = np.array([per[e][rule]["fnr"] for e in per])\n                    a = None if rule == "default" else float(rule.split("_")[1])\n                    summ[rule] = {"mean_fnr": float(f.mean()), "max_fnr": float(f.max()), "sd_fnr": float(f.std()),\n                                  "meeting": None if a is None else int((f <= a).sum()),\n                                  "mean_iou": float(np.mean([per[e][rule]["iou"] for e in per]))}\n                block[m] = {"n": len(per), "summary": summ,\n                            "events": {e: {r: {k: round(float(x), 4) for k, x in d.items()} for r, d in pe.items()} for e, pe in per.items()}}\n            if True:\n                ot = [data[f"otsu|{e}"][sub] for e in events if f"otsu|{e}" in data]\n                if ot:\n                    ious = [c[0] / max(1, c.sum()) for c in ot]\n                    fnrs = [c[2] / max(1, c[0] + c[2]) for c in ot]\n                    block["otsu"] = {"n": len(ot), "summary": {"default": {"mean_fnr": float(np.mean(fnrs)), "max_fnr": float(np.max(fnrs)),\n                                     "sd_fnr": float(np.std(fnrs)), "meeting": None, "mean_iou": float(np.mean(ious))}}}\n            out[f"{v}|{sub_name}"] = block\n    return out\n\n\ndef analyse(args):\n    data = dict(np.load(os.path.join(args.runs, f"tta_hist{tag(args)}.npz")))\n    report(compute(data), os.path.join(args.runs, f"tta_results{tag(args)}.json"),\n           os.path.join(args.runs, f"tta_summary{tag(args)}.md"))\n\n\ndef report(out, json_path, md_path):\n    lines = []\n    for key, block in out.items():\n            v, sub_name = key.split("|")\n            lines += [f"### {v}, {sub_name} chips", "",\n                      "| Adaptation | Rule | Mean miss rate | Worst flood | SD across floods | Floods meeting target | Mean IoU |",\n                      "| --- | --- | --- | --- | --- | --- | --- |"]\n            for m, b in block.items():\n                for rule, d in b["summary"].items():\n                    meet = "-" if d["meeting"] is None else f"{d[\'meeting\']}/{b[\'n\']}"\n                    lines.append(f"| {m} | {rule} | {d[\'mean_fnr\']:.3f} | {d[\'max_fnr\']:.3f} | {d[\'sd_fnr\']:.3f} | {meet} | {d[\'mean_iou\']:.3f} |")\n            lines.append("")\n            if "phys" in block and "none" in block:\n                lines += [f"Per flood at the 20% target ({v}, {sub_name}): miss rate / IoU", "",\n                          "| Flood | " + " | ".join(m for m in ALL_METHODS if m in block) + " |",\n                          "| --- |" + " --- |" * len([m for m in ALL_METHODS if m in block])]\n                for e in block["none"]["events"]:\n                    cells = []\n                    for m in ALL_METHODS:\n                        if m in block and e in block[m]["events"]:\n                            d = block[m]["events"][e]["crc_0.2"]\n                            cells.append(f"{d[\'fnr\']:.3f} / {d[\'iou\']:.3f}")\n                        elif m in block:\n                            cells.append("-")\n                    lines.append(f"| {e} | " + " | ".join(cells) + " |")\n                lines.append("")\n    with open(json_path, "w") as f:\n        json.dump(out, f, indent=1)\n    with open(md_path, "w") as f:\n        f.write("\\n".join(lines))\n    print("\\n".join(lines))\n    print("TTA_DONE")\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", default="data")\n    ap.add_argument("--runs", default="runs_loeo")\n    ap.add_argument("--analyse-only", action="store_true")\n    ap.add_argument("--steps", type=int, default=STEPS)\n    ap.add_argument("--seed", type=int, default=0, help="adaptation seed")\n    ap.add_argument("--model-seed", type=int, default=0, help="seed of the leave-one-event-out models to adapt")\n    ap.add_argument("--methods", default=",".join(METHODS), help="comma-separated, from " + ",".join(ALL_METHODS))\n    ap.add_argument("--no-analyse", action="store_true")\n    a = ap.parse_args()\n    a.methods = a.methods.split(",")\n    if not a.analyse_only:\n        phase1(a)\n    if not a.no_analyse:\n        analyse(a)\n', 'ks_eval.py': '"""\nPX1 stage 6: cross-dataset test on Kuro Siwo (up to 43 flood events, none seen in training).\n\nModels: the Sen1Floods11 models trained on the standard training split (runs/V0_seed*, V1_seed*).\nEvery Kuro Siwo flood event is unseen. For each event we apply the same label-free\nadaptation methods as tta_px1.py (none, tent, st, phys) and store probability histograms;\nthen the miss-rate guarantee is calibrated leave-one-event-out across the Kuro Siwo events,\nso with ~40 events the guarantee is far less conservative than with 11.\n\nInputs per chip: post-flood VV, VH (dB) and the Kuro Siwo DEM (10 m grid). Labels: permanent\nwater and flood water both count as water; class 3 and invalid pixels are ignored.\n\nUsage:  python ks_eval.py --data data --ks data_ks --runs runs --model-seed 0 --out runs_ks\n        python ks_eval.py --analyse-only --model-seed 0\n"""\nimport argparse\nimport csv\nimport glob\nimport os\nimport time\n\nos.environ.setdefault("GDAL_CACHEMAX", "64")\n\nimport numpy as np\n\nfrom tta_px1 import BINS, COMBOS, METHODS, VARIANTS, hand_channel, compute, curves, otsu_counts, report, run_one, save\n\nMIN_CHIPS = 8\nV = "_v2"  # v2: events merged across Kuro Siwo splits (v1 dropped the test chips of events in both)\n\n\ndef terrain_10m(dem):\n    gy, gx = np.gradient(dem.astype(np.float64), 10.0, 10.0)\n    slope = np.degrees(np.arctan(np.hypot(gx, gy)))\n    relh = dem - np.percentile(dem, 2)\n    return np.stack([np.clip(relh / 50.0, 0, 2), np.clip(slope / 10.0, 0, 3)]).astype(np.float32)\n\n\ndef load_ks_event(files):\n    ev = {"raw": [], "terr": [], "lab": [], "clean": []}\n    for f in files:\n        z = np.load(f)\n        raw = np.clip(np.stack([z["vv"], z["vh"]]).astype(np.float32), -50, 5)\n        dem = z["dem"].astype(np.float32)\n        if not np.all(np.isfinite(dem)):\n            dem = np.nan_to_num(dem, nan=float(np.nanmedian(dem)) if np.isfinite(dem).any() else 0.0)\n        m = z["mask"]\n        lab = np.full(m.shape, 255, np.uint8)\n        lab[m == 0] = 0\n        lab[(m == 1) | (m == 2)] = 1\n        lab[(z["valid"] == 0) | ~np.isfinite(z["vv"]) | ~np.isfinite(z["vh"])] = 255\n        raw = np.nan_to_num(raw, nan=-50.0)\n        ev["raw"].append(raw)\n        hp = f.replace("data_ks", "data_ks_hand", 1)[:-4] + ".npy"\n        hand = np.load(hp) if os.path.exists(hp) else None\n        ev["hand_ok"] = ev.get("hand_ok", 0) + (hand is not None)\n        ev["terr"].append(hand_channel(hand, terrain_10m(dem)))\n        ev["lab"].append(lab)\n        ev["clean"].append(False)\n    return ev\n\n\ndef events_of(ks):\n    """Event id -> chip files, merged across the test_GRD and train_GRD folders (an event can be in both)."""\n    ev = {}\n    for d in sorted(glob.glob(os.path.join(ks, "*_GRD", "*"))):\n        ev.setdefault(os.path.basename(d), []).extend(sorted(glob.glob(os.path.join(d, "*.npz"))))\n    return {e: f for e, f in sorted(ev.items()) if len(f) >= MIN_CHIPS}\n\n\ndef phase1(args):\n    import torch\n    from sen1floods11_unet_baseline import channel_stats, read_split\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    os.makedirs(args.out, exist_ok=True)\n    path = os.path.join(args.out, f"ks_hist_m{args.model_seed}{V}.npz")\n    done = dict(np.load(path)) if os.path.exists(path) else {}\n    mean, std = channel_stats(args.data, read_split(args.data, "train"))\n    events = events_of(args.ks)\n    print(f"{len(events)} Kuro Siwo events with >= {MIN_CHIPS} chips", flush=True)\n    for e, files in events.items():\n        todo = [(m, v) for v in VARIANTS for m in args.methods if f"{m}|{v}|{e}" not in done]\n        if not todo and f"otsu|{e}" in done:\n            continue\n        ev = load_ks_event(files)\n        if f"otsu|{e}" not in done:\n            done[f"otsu|{e}"], thr = otsu_counts(ev)\n        for m, v in todo:\n            t0 = time.time()\n            ckpt = os.path.join(args.runs, f"{v}_seed{args.model_seed}", "model.pt")\n            h = run_one(m, v, e, ev, mean, std, args, device, ckpt=ckpt)\n            done[f"{m}|{v}|{e}"] = h\n            fnr, _, iou = curves(h[0])\n            print(f"{m:10s} {v} event {e:>4s} ({len(files)} chips): miss rate at 0.5 {fnr[BINS // 2]:.3f}, "\n                  f"IoU {iou[BINS // 2]:.3f}, {time.time() - t0:.0f}s", flush=True)\n            save(path, done)\n        save(path, done)\n        if device == "cuda":\n            torch.cuda.empty_cache()\n\n\ndef analyse(args):\n    data = dict(np.load(os.path.join(args.out, f"ks_hist_m{args.model_seed}{V}.npz")))\n    events = sorted({k.split("|")[2] for k in data if k.count("|") == 2})\n    report(compute(data, events), os.path.join(args.out, f"ks_results_m{args.model_seed}{V}.json"),\n           os.path.join(args.out, f"ks_summary_m{args.model_seed}{V}.md"))\n    print("KS_EVAL_DONE", flush=True)\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", default="data")\n    ap.add_argument("--ks", default="data_ks")\n    ap.add_argument("--runs", default="runs")\n    ap.add_argument("--out", default="runs_ks")\n    ap.add_argument("--model-seed", type=int, default=0)\n    ap.add_argument("--seed", type=int, default=0)\n    ap.add_argument("--steps", type=int, default=200)\n    ap.add_argument("--analyse-only", action="store_true")\n    ap.add_argument("--methods", default=",".join(METHODS + COMBOS))\n    a = ap.parse_args()\n    a.methods = a.methods.split(",")\n    if not a.analyse_only:\n        phase1(a)\n    analyse(a)\n', 'r34_patch.py': '"""\nSecond backbone (ResNet-34 U-Net, ImageNet-pretrained encoder) for PX1, as small exact patches to the existing code.\n\nThe architecture is chosen by the environment variable PX1_ARCH (unset or "unet" = the original U-Net, "r34" =\nsegmentation_models_pytorch Unet with a ResNet-34 encoder). Every patch is checked to match exactly once, so a\nmismatch with the bundle\'s code fails immediately instead of silently doing something else.\n\nAlso adds --variants, --part and --nparts to ks_eval.py so Kuro Siwo can be split across GPUs and run on V0 only.\n\nUsage: python r34_patch.py <code dir> [--check]\n"""\nimport os\nimport sys\n\nBUILD = \'\'\'\n\ndef build_model(cin=2, pretrained=False):\n    """PX1_ARCH=unet (default): the original U-Net. PX1_ARCH=r34: U-Net with a ResNet-34 encoder (ImageNet weights when training)."""\n    arch = os.environ.get("PX1_ARCH", "unet")\n    if arch == "unet":\n        return UNet(cin=cin)\n    import segmentation_models_pytorch as smp\n    enc = {"r34": "resnet34", "r50": "resnet50"}[arch]\n    return smp.Unet(enc, encoder_weights="imagenet" if pretrained else None, in_channels=cin, classes=2)\n\n\'\'\'\n\nPATCHES = {\n    "sen1floods11_unet_baseline.py": [\n        ("\\n\\n# ---------------------------------------------------------------- evaluation\\n",\n         BUILD + "\\n# ---------------------------------------------------------------- evaluation\\n"),\n    ],\n    "train_px1.py": [\n        ("from sen1floods11_unet_baseline import IGNORE, UNet, channel_stats,",\n         "from sen1floods11_unet_baseline import IGNORE, UNet, build_model, channel_stats,"),\n        ("    model = UNet(cin=4 if use_dem else 2).to(device)\\n",\n         "    model = build_model(cin=4 if use_dem else 2, pretrained=True).to(device)\\n"),\n    ],\n    "tta_px1.py": [\n        ("    from sen1floods11_unet_baseline import IGNORE, UNet\\n",\n         "    from sen1floods11_unet_baseline import IGNORE, UNet, build_model\\n"),\n        ("    model = UNet(cin=4 if use_dem else 2).to(device)\\n",\n         "    model = build_model(cin=4 if use_dem else 2).to(device)\\n"),\n    ],\n    "ks_eval.py": [\n        ("    events = events_of(args.ks)\\n",\n         "    events = dict(list(events_of(args.ks).items())[args.part::args.nparts])\\n"),\n        ("        todo = [(m, v) for v in VARIANTS for m in args.methods",\n         "        todo = [(m, v) for v in args.variants for m in args.methods"),\n        ("    a.methods = a.methods.split(\\",\\")\\n",\n         "    a.methods = a.methods.split(\\",\\")\\n    a.variants = a.variants.split(\\",\\")\\n"),\n        ("    ap.add_argument(\\"--analyse-only\\", action=\\"store_true\\")\\n",\n         "    ap.add_argument(\\"--analyse-only\\", action=\\"store_true\\")\\n    ap.add_argument(\\"--variants\\", default=\\"V0,V1\\")\\n"\n         "    ap.add_argument(\\"--part\\", type=int, default=0)\\n    ap.add_argument(\\"--nparts\\", type=int, default=1)\\n"),\n    ],\n}\n\n\ndef main():\n    code, check = sys.argv[1], "--check" in sys.argv\n    for f, ps in PATCHES.items():\n        p = os.path.join(code, f)\n        s = open(p).read()\n        for old, new in ps:\n            n = s.count(old)\n            if n != 1:\n                raise SystemExit(f"patch mismatch in {f}: {old.strip()[:60]!r} found {n} times")\n            s = s.replace(old, new)\n        if not check:\n            open(p, "w").write(s)\n        print(f"{\'checked\' if check else \'patched\'} {f} ({len(ps)} changes)", flush=True)\n    print("PATCH_OK", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'rev_patch.py': '"""\nRevision patch (applied after r34_patch.py): adds three published test-time adaptation baselines to tta_px1.run_one\nand a second label definition for Kuro Siwo.\n\n  eata   EATA (Niu et al., 2022): entropy minimisation on batch-norm weights, only on reliable pixels\n         (entropy < 0.4 ln 2), re-weighted by exp(E0 - entropy), with a Fisher penalty that anchors the weights.\n         The Fisher information is estimated with the unadapted model on the first 20 target batches.\n  sar    SAR (Niu et al., 2023): sharpness-aware entropy minimisation on batch-norm weights, reliable pixels only.\n  cotta  CoTTA (Wang et al., 2022): mean teacher, augmentation-averaged teacher predictions, all weights updated,\n         stochastic restore of 1% of the weights to their source values at every step.\n\nAll three use the same crops, batch size, number of steps and optimiser as the TENT baseline.\n\nKuro Siwo: histogram slot 1 (unused for Kuro Siwo) now holds the scores with permanent water ignored\n(only flood water counts as water), so both label definitions come from one run.\n"""\nimport os\nimport sys\n\nNEW = \'\'\'    if method in ("eata", "sar"):\n        import math\n        E0 = 0.4 * math.log(2)\n        model.train()\n        params = [p for mod in model.modules() if isinstance(mod, torch.nn.BatchNorm2d) for p in (mod.weight, mod.bias)]\n        for p in model.parameters():\n            p.requires_grad_(False)\n        for p in params:\n            p.requires_grad_(True)\n\n        def ent(r, t):\n            p = model(inp(r, t)).softmax(1)\n            return -(p * torch.log(p.clamp_min(1e-6))).sum(1)\n\n        opt = torch.optim.Adam(params, lr=LR_TENT)\n        if method == "eata":\n            anchor = [p.detach().clone() for p in params]\n            fisher = [torch.zeros_like(p) for p in params]\n            for _ in range(20):\n                r, t = crops()\n                logits = model(inp(r, t))\n                g = torch.autograd.grad(F.cross_entropy(logits, logits.argmax(1)), params)\n                for f, gi in zip(fisher, g):\n                    f += gi.detach() ** 2 / 20\n        sel_share = []\n        for _ in range(args.steps):\n            r, t = crops()\n            H = ent(r, t)\n            sel = (H < E0).detach()\n            sel_share.append(float(sel.float().mean()))\n            if method == "eata":\n                w = torch.exp(E0 - H.detach())\n                loss = (H * w * sel).sum() / sel.sum().clamp_min(1)\n                loss = loss + 2000.0 * sum((f * (p - a) ** 2).sum() for f, p, a in zip(fisher, params, anchor))\n                opt.zero_grad(); loss.backward(); opt.step()\n            else:\n                loss = (H * sel).sum() / sel.sum().clamp_min(1)\n                opt.zero_grad(); loss.backward()\n                gn = torch.sqrt(sum((p.grad ** 2).sum() for p in params)) + 1e-12\n                eps = [0.05 * p.grad / gn for p in params]\n                with torch.no_grad():\n                    for p, e_ in zip(params, eps):\n                        p.add_(e_)\n                H2 = ent(r, t)\n                sel2 = sel & (H2 < E0).detach()\n                loss2 = (H2 * sel2).sum() / sel2.sum().clamp_min(1)\n                opt.zero_grad(); loss2.backward()\n                with torch.no_grad():\n                    for p, e_ in zip(params, eps):\n                        p.sub_(e_)\n                opt.step()\n        print(f"    reliable share of pixels {np.mean(sel_share):.3f}", flush=True)\n        for p in model.parameters():\n            p.requires_grad_(True)\n    cotta_teacher = None\n    if method == "cotta":\n        source = {k: p.detach().clone() for k, p in model.named_parameters()}\n        cotta_teacher = copy.deepcopy(model).train()\n        for p in cotta_teacher.parameters():\n            p.requires_grad_(False)\n        model.train()\n        opt = torch.optim.Adam(model.parameters(), lr=LR_ST)\n        for _ in range(args.steps):\n            r, t = crops()\n            with torch.no_grad():\n                acc = 0\n                for k in range(4):\n                    off = (torch.rand(r.shape[0], 1, 1, 1, device=device) * 3 - 1.5) if k else 0.0\n                    x = inp(r + off, t)\n                    o = cotta_teacher(x.flip(3)).softmax(1).flip(3) if k % 2 else cotta_teacher(x).softmax(1)\n                    acc = acc + o / 4\n            loss = -(acc * F.log_softmax(model(inp(r, t)), 1)).sum(1).mean()\n            opt.zero_grad(); loss.backward(); opt.step()\n            with torch.no_grad():\n                for pt_, ps_ in zip(cotta_teacher.parameters(), model.parameters()):\n                    pt_.mul_(EMA).add_(ps_.detach(), alpha=1 - EMA)\n                for k, p in model.named_parameters():\n                    mask = torch.rand_like(p) < 0.01\n                    p.copy_(torch.where(mask, source[k], p))\n    final = model\n    if cotta_teacher is not None:\n        final = cotta_teacher\n\'\'\'\n\nHIST_OLD = \'\'\'            for sub in ([0, 1] if ev["clean"][i] else [0]):\n                for c in (0, 1):\n                    h[sub, c] += torch.bincount(b[lab == c], minlength=BINS).cpu().numpy()\n\'\'\'\nHIST_NEW = HIST_OLD + \'\'\'            if "lab_alt" in ev:\n                lab = torch.from_numpy(ev["lab_alt"][i].astype(np.int64)).to(device)\n                for c in (0, 1):\n                    h[1, c] += torch.bincount(b[lab == c], minlength=BINS).cpu().numpy()\n\'\'\'\n\nPATCHES = {\n    "tta_px1.py": [\n        ("ALL_METHODS = METHODS + ABLATIONS + COMBOS + ADAPTIVE\\n",\n         \'ALL_METHODS = METHODS + ABLATIONS + COMBOS + ADAPTIVE + ["eata", "sar", "cotta", "ft"]\\n\'),\n        ("    final = model\\n    if base in (", NEW + "    if base in ("),\n        (HIST_OLD, HIST_NEW),\n    ],\n    "ks_eval.py": [\n        (\'        ev["lab"].append(lab)\\n\',\n         \'        ev["lab"].append(lab)\\n        alt = lab.copy()\\n        alt[(m == 1) & (lab != 255)] = 255\\n\'\n         \'        ev.setdefault("lab_alt", []).append(alt)\\n\'),\n    ],\n}\n\n\ndef main():\n    code, check = sys.argv[1], "--check" in sys.argv\n    for f, ps in PATCHES.items():\n        p = os.path.join(code, f)\n        s = open(p).read()\n        for old, new in ps:\n            n = s.count(old)\n            if n != 1:\n                raise SystemExit(f"patch mismatch in {f}: {old.strip()[:60]!r} found {n} times")\n            s = s.replace(old, new)\n        if not check:\n            open(p, "w").write(s)\n        print(f"{\'checked\' if check else \'patched\'} {f} ({len(ps)} changes)", flush=True)\n    print("REV_PATCH_OK", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'mm_eval.py': '"""\nMMFlood final test (pre-registered 9 Nov 2026). Same adaptation code and settings as ks_eval.py.\n\nPer event (chips from mm_prep.py) and model, store probability histograms for each method; the\nanalysis (mm_analyse.py) pools the parts and models and runs the pre-registered tests.\n\nUsage:  python mm_eval.py --data data --mm data_mm --runs runs --model-seed 0 --part 0 --nparts 2 --out runs_mm\n"""\nimport argparse\nimport csv\nimport os\nimport time\n\nos.environ.setdefault("GDAL_CACHEMAX", "64")\nimport numpy as np\n\nfrom ks_eval import terrain_10m\nfrom tta_px1 import BINS, curves, hand_channel, otsu_counts, run_one, save\n\nMETHODS_MM = ["none", "tent", "adabn", "adabn_phys", "adabn_physa"]\nMIN_CHIPS = 8\n\n\ndef load_mm_event(files):\n    ev = {"raw": [], "terr": [], "lab": [], "clean": []}\n    for f in files:\n        z = np.load(f)\n        ev["raw"].append(np.clip(np.stack([z["vv"], z["vh"]]).astype(np.float32), -50, 5))\n        hand = z["hand"]\n        ev["hand_ok"] = ev.get("hand_ok", 0) + int(np.any(hand != 0))\n        ev["terr"].append(hand_channel(hand, terrain_10m(z["dem"])))\n        ev["lab"].append(z["lab"])\n        ev["clean"].append(False)\n    return ev\n\n\ndef events_of(mm):\n    ev = {}\n    for r in csv.DictReader(open(os.path.join(mm, "index.csv"))):\n        ev.setdefault(r["event"], []).append(r["file"])\n    return {e: sorted(f) for e, f in sorted(ev.items()) if len(f) >= MIN_CHIPS}\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", default="data")\n    ap.add_argument("--mm", default="data_mm")\n    ap.add_argument("--runs", default="runs")\n    ap.add_argument("--out", default="runs_mm")\n    ap.add_argument("--model-seed", type=int, default=0)\n    ap.add_argument("--seed", type=int, default=0)\n    ap.add_argument("--steps", type=int, default=200)\n    ap.add_argument("--part", type=int, default=0)\n    ap.add_argument("--nparts", type=int, default=1)\n    ap.add_argument("--variants", default="V0")\n    ap.add_argument("--methods", default=",".join(METHODS_MM))\n    a = ap.parse_args()\n    a.methods, variants = a.methods.split(","), a.variants.split(",")\n\n    import torch\n    from sen1floods11_unet_baseline import channel_stats, read_split\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    os.makedirs(a.out, exist_ok=True)\n    path = os.path.join(a.out, f"mm_hist_m{a.model_seed}_p{a.part}.npz")\n    done = dict(np.load(path)) if os.path.exists(path) else {}\n    mean, std = channel_stats(a.data, read_split(a.data, "train"))\n    events = list(events_of(a.mm).items())[a.part::a.nparts]\n    print(f"part {a.part}/{a.nparts}: {len(events)} MMFlood events", flush=True)\n    for e, files in events:\n        todo = [(m, v) for v in variants for m in a.methods if f"{m}|{v}|{e}" not in done]\n        if not todo and f"otsu|{e}" in done:\n            continue\n        ev = load_mm_event(files)\n        if f"otsu|{e}" not in done:\n            done[f"otsu|{e}"], _ = otsu_counts(ev)\n        for m, v in todo:\n            t0 = time.time()\n            ckpt = os.path.join(a.runs, f"{v}_seed{a.model_seed}", "model.pt")\n            h = run_one(m, v, e, ev, mean, std, a, device, ckpt=ckpt)\n            done[f"{m}|{v}|{e}"] = h\n            fnr, _, iou = curves(h[0])\n            print(f"{m:11s} {v} event {e} ({len(files)} chips): miss rate at 0.5 {fnr[BINS // 2]:.3f}, IoU {iou[BINS // 2]:.3f}, "\n                  f"{time.time() - t0:.0f}s", flush=True)\n            save(path, done)\n        save(path, done)\n        if device == "cuda":\n            torch.cuda.empty_cache()\n    print("MM_EVAL_DONE", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'mm_prep.py': '"""\nMMFlood final-test preparation (pre-registered 9 Nov 2026; see the PX1 doc).\n\n  1. download mmflood.zip (Zenodo 6534637, 11.3 GB) and extract\n  2. drop events overlapping Kuro Siwo (300 km and 365 days) or Sen1Floods11 (300 km, same or adjacent year)\n  3. per image: dB conversion if linear, resample 20 m -> 10 m, cut 512 x 512 chips with >= 50% valid radar\n  4. at most 48 chips per event, spread evenly over the event\'s chips; events with < 8 chips dropped\n  5. labels: 1 water, 0 dry, 255 ignore (invalid radar; OSM permanent water that the mask calls dry)\n  6. HAND from GLO-30 HAND on each chip grid\n\nWrites data_mm/<event>/<chip>.npz (vv, vh dB float16; dem float32; hand float32; lab uint8),\ndata_mm/index.csv and data_mm/excluded.csv. Prints MM_PREP_DONE.\n"""\nimport argparse\nimport csv\nimport glob\nimport json\nimport math\nimport os\nimport subprocess\nfrom concurrent.futures import ThreadPoolExecutor\n\nos.environ.setdefault("GDAL_CACHEMAX", "256")\nimport numpy as np\nimport rasterio\nfrom rasterio.transform import Affine\nfrom rasterio.warp import Resampling, reproject, transform_bounds\n\nURL = "https://zenodo.org/records/6534637/files/mmflood.zip?download=1"\nCHIP, MAX_CHIPS, MIN_CHIPS, MIN_VALID = 512, 48, 8, 0.5\nS1F_YEARS = {"Bolivia": 2018, "Ghana": 2018, "India": 2016, "Mekong": 2018, "Nigeria": 2018, "Pakistan": 2017,\n             "Paraguay": 2018, "Somalia": 2018, "Spain": 2019, "Sri-Lanka": 2017, "USA": 2019}\n\n\ndef km(lon1, lat1, lon2, lat2):\n    p1, p2 = math.radians(lat1), math.radians(lat2)\n    a = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lon2 - lon1) / 2) ** 2\n    return 6371 * 2 * math.asin(math.sqrt(min(1, a)))\n\n\ndef s1f_centroids(data):\n    pts = {}\n    for f in glob.glob(os.path.join(data, "S1Hand", "*.tif")):\n        e = os.path.basename(f).split("_")[0]\n        with rasterio.open(f) as src:\n            b = src.bounds if src.crs.is_geographic else transform_bounds(src.crs, "EPSG:4326", *src.bounds)\n        pts.setdefault(e, []).append(((b[0] + b[2]) / 2, (b[1] + b[3]) / 2))\n    return {e: (np.mean([p[0] for p in v]), np.mean([p[1] for p in v])) for e, v in pts.items()}\n\n\ndef download(root):\n    z = os.path.join(root, "mmflood.zip")\n    if not os.path.exists(os.path.join(root, "extracted.ok")):\n        os.makedirs(root, exist_ok=True)\n        for t in range(4):\n            if subprocess.call(["wget", "-q", "-c", "-O", z, URL]) == 0:\n                break\n        print(f"downloaded {os.path.getsize(z) / 2**30:.1f} GB", flush=True)\n        subprocess.check_call(["unzip", "-q", "-o", z, "-d", root])\n        os.remove(z)\n        open(os.path.join(root, "extracted.ok"), "w").close()\n    for p in sorted(glob.glob(os.path.join(root, "*")) + glob.glob(os.path.join(root, "*", "*")))[:20]:\n        print(f"  layout: {p}", flush=True)\n    if not glob.glob(os.path.join(root, "**", "s1_raw"), recursive=True):  # archives inside the archive\n        for z in glob.glob(os.path.join(root, "**", "*.zip"), recursive=True) + glob.glob(os.path.join(root, "**", "*.tar*"), recursive=True):\n            print(f"  extracting inner archive {z}", flush=True)\n            if z.endswith(".zip"):\n                subprocess.call(["unzip", "-q", "-o", z, "-d", os.path.dirname(z)])\n            else:\n                subprocess.call(["tar", "-xf", z, "-C", os.path.dirname(z)])\n    acts = glob.glob(os.path.join(root, "**", "activations.json"), recursive=True)\n    # event folders = parents of every s1_raw folder, wherever the zip put them\n    dirs = {os.path.basename(os.path.dirname(p)): os.path.dirname(p)\n            for p in glob.glob(os.path.join(root, "**", "s1_raw"), recursive=True) if os.path.isdir(p)}\n    print(f"activations.json: {acts[:2]}; {len(dirs)} event folders", flush=True)\n    if not dirs:\n        raise SystemExit("no s1_raw folders found in the extracted archive")\n    return (acts[0] if acts else None), dirs\n\n\ndef to_db(a):\n    a = a.astype(np.float32)\n    v = a[np.isfinite(a) & (a != 0)]\n    if v.size and np.median(v) > 0:  # linear power -> dB\n        a = np.where(a > 0, 10 * np.log10(np.maximum(a, 1e-6)), np.nan)\n    return a\n\n\ndef resample(src, band, transform, crs, shape, method):\n    out = np.full(shape, np.nan, np.float32)\n    reproject(src.read(band).astype(np.float32), out, src_transform=src.transform, src_crs=src.crs,\n              dst_transform=transform, dst_crs=crs, src_nodata=src.nodata, dst_nodata=np.nan, resampling=method)\n    return out\n\n\ndef image_chips(base, name):\n    """All valid 512 x 512 chips of one MMFlood image, at 10 m."""\n    with rasterio.open(os.path.join(base, "s1_raw", name)) as s1:\n        crs, t = s1.crs, s1.transform\n        lat = (s1.bounds.top + s1.bounds.bottom) / 2\n        res_m = abs(t.a) * (111320 * math.cos(math.radians(lat)) if crs.is_geographic else 1)\n        f = max(1.0, res_m / 10.0)\n        shape = (int(s1.height * f), int(s1.width * f))\n        tt = t * Affine.scale(1 / f)\n        vv = to_db(resample(s1, 1, tt, crs, shape, Resampling.bilinear))\n        vh = to_db(resample(s1, 2, tt, crs, shape, Resampling.bilinear))\n    with rasterio.open(os.path.join(base, "mask", name)) as m:\n        mask = resample(m, 1, tt, crs, shape, Resampling.nearest)\n    with rasterio.open(os.path.join(base, "DEM", name)) as d:\n        dem = resample(d, 1, tt, crs, shape, Resampling.bilinear)\n    hp = os.path.join(base, "hydro", name)\n    hydro = None\n    if os.path.exists(hp):\n        with rasterio.open(hp) as h:\n            hydro = resample(h, 1, tt, crs, shape, Resampling.nearest)\n    valid = np.isfinite(vv) & np.isfinite(vh) & (vv > -49) & (vh > -49)\n    vals = set(np.unique(mask[np.isfinite(mask)]).astype(int).tolist())\n    water_val = 255 if vals <= {0, 255} and 255 in vals else 1  # masks stored as 0/255 or 0/1(/255 nodata)\n    lab = np.full(shape, 255, np.uint8)\n    lab[(mask == 0) & valid] = 0\n    lab[(mask == water_val) & valid] = 1\n    if hydro is not None:\n        lab[(hydro > 0) & (lab == 0)] = 255\n    chips = []\n    for r in range(0, shape[0] - CHIP + 1, CHIP):\n        for c in range(0, shape[1] - CHIP + 1, CHIP):\n            v = valid[r:r + CHIP, c:c + CHIP]\n            if v.mean() < MIN_VALID:\n                continue\n            d = dem[r:r + CHIP, c:c + CHIP]\n            d = np.nan_to_num(d, nan=float(np.nanmedian(d)) if np.isfinite(d).any() else 0.0)\n            chips.append({"vv": np.nan_to_num(vv[r:r + CHIP, c:c + CHIP], nan=-50.0), "vh": np.nan_to_num(vh[r:r + CHIP, c:c + CHIP], nan=-50.0),\n                          "dem": d, "lab": lab[r:r + CHIP, c:c + CHIP], "transform": tuple(tt * Affine.translation(c, r))[:6], "crs": crs.to_wkt()})\n    return chips\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--raw", default="mmflood_raw")\n    ap.add_argument("--out", default="data_mm")\n    ap.add_argument("--data", default="data")\n    ap.add_argument("--ks-events", default="ks_events.json")\n    ap.add_argument("--workers", type=int, default=4)\n    a = ap.parse_args()\n    meta_path, dirs = download(a.raw)\n    meta = {}\n    if meta_path:\n        txt = open(meta_path).read()\n        try:\n            meta = json.loads(txt)\n        except json.JSONDecodeError:  # tolerate trailing commas\n            import re\n            meta = json.loads(re.sub(r",\\s*([}\\]])", r"\\1", txt))\n    if isinstance(meta, list):\n        meta = {m.get("code") or m.get("activation") or str(i): m for i, m in enumerate(meta)}\n    acts = {}\n    for d in sorted(dirs):\n        info = meta.get(d) or meta.get(d.split("-")[0]) or {}\n        if "lon" not in info or "lat" not in info:  # fall back to the first image\'s centre\n            tifs = sorted(glob.glob(os.path.join(dirs[d], "s1_raw", "*.tif")))\n            if tifs:\n                with rasterio.open(tifs[0]) as src:\n                    b = src.bounds if src.crs.is_geographic else transform_bounds(src.crs, "EPSG:4326", *src.bounds)\n                info = dict(info, lon=(b[0] + b[2]) / 2, lat=(b[1] + b[3]) / 2)\n        acts[d] = {"lon": info.get("lon", 0), "lat": info.get("lat", 0), "start": str(info.get("start") or info.get("end") or "2000-01-01"),\n                   "country": info.get("country"), "subset": info.get("subset")}\n    print(f"metadata example: {next(iter(meta.items()), None)}", flush=True)\n    ks = json.load(open(a.ks_events))\n    s1f = s1f_centroids(a.data)\n    keep, excl = [], []\n    for code, info in sorted(acts.items()):\n        lon, lat, date = float(info["lon"]), float(info["lat"]), info["start"][:10]\n        why = []\n        for k, e in ks.items():\n            if km(lon, lat, e["lon"], e["lat"]) <= 300:\n                dd = abs((np.datetime64(date) - np.datetime64(e["date"])).astype(int))\n                if dd <= 365:\n                    why.append(f"KuroSiwo {k} ({dd} days)")\n        for e, (elon, elat) in s1f.items():\n            if km(lon, lat, elon, elat) <= 300 and abs(int(date[:4]) - S1F_YEARS.get(e, 0)) <= 1:\n                why.append(f"Sen1Floods11 {e}")\n        (excl if why else keep).append((code, info, "; ".join(why)))\n    print(f"{len(acts)} activations; {len(excl)} excluded for overlap; {len(keep)} kept", flush=True)\n    os.makedirs(a.out, exist_ok=True)\n    with open(os.path.join(a.out, "excluded.csv"), "w", newline="") as f:\n        w = csv.writer(f)\n        w.writerow(["event", "country", "start", "reason"])\n        for code, info, why in excl:\n            w.writerow([code, info.get("country"), info["start"][:10], why])\n\n    import hand_fetch\n    hand_fetch.probe()\n    rows = []\n\n    def do_event(item):\n        code, info, _ = item\n        base = dirs[code]\n        if not os.path.isdir(base):\n            return []\n        evdir = os.path.join(a.out, code)\n        if os.path.exists(os.path.join(evdir, "done.ok")):\n            return [r for r in csv.DictReader(open(os.path.join(evdir, "chips.csv")))]\n        chips = []\n        for name in sorted(os.listdir(os.path.join(base, "s1_raw"))):\n            try:\n                chips += [(name, ch) for ch in image_chips(base, name)]\n            except Exception as e:\n                print(f"  {code}/{name}: {type(e).__name__} {e}", flush=True)\n        if len(chips) < MIN_CHIPS:\n            return []\n        if len(chips) > MAX_CHIPS:\n            idx = np.linspace(0, len(chips) - 1, MAX_CHIPS).round().astype(int)\n            chips = [chips[i] for i in sorted(set(idx))]\n        os.makedirs(evdir, exist_ok=True)\n        out = []\n        for i, (name, ch) in enumerate(chips):\n            tr = Affine(*ch["transform"])\n            hand = hand_fetch.hand_on_grid(tr, rasterio.crs.CRS.from_wkt(ch["crs"]), (CHIP, CHIP))\n            fn = os.path.join(evdir, f"{i:03d}.npz")\n            np.savez_compressed(fn, vv=ch["vv"].astype(np.float16), vh=ch["vh"].astype(np.float16), dem=ch["dem"].astype(np.float32),\n                                hand=hand.astype(np.float32), lab=ch["lab"])\n            out.append({"event": code, "file": fn, "image": name, "country": info.get("country"), "start": info["start"][:10],\n                        "water_px": int((ch["lab"] == 1).sum()), "dry_px": int((ch["lab"] == 0).sum())})\n        with open(os.path.join(evdir, "chips.csv"), "w", newline="") as f:\n            w = csv.DictWriter(f, fieldnames=list(out[0].keys()))\n            w.writeheader(); w.writerows(out)\n        open(os.path.join(evdir, "done.ok"), "w").close()\n        print(f"  {code} ({info.get(\'country\')}): {len(out)} chips, water px {sum(r[\'water_px\'] for r in out)}, dry px {sum(r[\'dry_px\'] for r in out)}", flush=True)\n        return out\n\n    with ThreadPoolExecutor(a.workers) as ex:\n        for r in ex.map(do_event, keep):\n            rows += r\n    with open(os.path.join(a.out, "index.csv"), "w", newline="") as f:\n        w = csv.DictWriter(f, fieldnames=["event", "file", "image", "country", "start", "water_px", "dry_px"])\n        w.writeheader(); w.writerows(rows)\n    n_ev = len({r["event"] for r in rows})\n    print(f"MMFlood: {n_ev} events, {len(rows)} chips", flush=True)\n    print("MM_PREP_DONE", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'ks_events.json': '{\n"118": {\n"lon": -2.333,\n"lat": 42.358,\n"date": "2015-02-02"\n},\n"130": {\n"lon": 94.896,\n"lat": 17.072,\n"date": "2015-08-01"\n},\n"147": {\n"lon": -2.821,\n"lat": 54.647,\n"date": "2015-12-05"\n},\n"267": {\n"lon": 21.491,\n"lat": 55.287,\n"date": "2018-01-31"\n},\n"273": {\n"lon": 19.546,\n"lat": 42.01,\n"date": "2018-03-10"\n},\n"275": {\n"lon": 16.674,\n"lat": 45.387,\n"date": "2018-03-21"\n},\n"277": {\n"lon": 26.462,\n"lat": 41.291,\n"date": "2018-03-27"\n},\n"279": {\n"lon": -0.248,\n"lat": 41.203,\n"date": "2018-04-12"\n},\n"321": {\n"lon": -86.991,\n"lat": 13.049,\n"date": "2018-10-06"\n},\n"324": {\n"lon": 2.674,\n"lat": 43.226,\n"date": "2018-10-15"\n},\n"417": {\n"lon": -8.638,\n"lat": 40.189,\n"date": "2019-12-19"\n},\n"421": {\n"lon": 2.96,\n"lat": 42.721,\n"date": "2020-01-22"\n},\n"427": {\n"lon": 13.141,\n"lat": 58.173,\n"date": "2020-02-18"\n},\n"437": {\n"lon": -0.093,\n"lat": 43.971,\n"date": "2020-05-10"\n},\n"445": {\n"lon": 26.705,\n"lat": 48.199,\n"date": "2020-06-26"\n},\n"470": {\n"lon": 0.573,\n"lat": 10.409,\n"date": "2020-10-05"\n},\n"492": {\n"lon": -0.982,\n"lat": 43.705,\n"date": "2021-01-01"\n},\n"497": {\n"lon": 6.651,\n"lat": 51.446,\n"date": "2021-02-01"\n},\n"498": {\n"lon": 1.842,\n"lat": 45.011,\n"date": "2021-02-02"\n},\n"502": {\n"lon": -8.449,\n"lat": 52.029,\n"date": "2021-02-23"\n},\n"518": {\n"lon": 5.632,\n"lat": 50.571,\n"date": "2021-07-14"\n},\n"555": {\n"lon": -1.848,\n"lat": 42.449,\n"date": "2021-12-10"\n},\n"567": {\n"lon": 152.999,\n"lat": -29.618,\n"date": "2022-02-26"\n},\n"1111004": {\n"lon": -96.354,\n"lat": 29.119,\n"date": "2017-08-28"\n},\n"1111005": {\n"lon": 48.402,\n"lat": -17.628,\n"date": "2020-01-25"\n},\n"1111006": {\n"lon": 46.6,\n"lat": -16.461,\n"date": "2020-01-30"\n},\n"1111007": {\n"lon": 85.67,\n"lat": 25.55,\n"date": "2019-09-17"\n},\n"1111008": {\n"lon": -83.538,\n"lat": 14.072,\n"date": "2020-08-15"\n},\n"1111009": {\n"lon": 68.295,\n"lat": 27.36,\n"date": "2022-09-10"\n},\n"1111010": {\n"lon": -80.712,\n"lat": -5.568,\n"date": "2017-03-10"\n},\n"1111011": {\n"lon": 121.698,\n"lat": 17.584,\n"date": "2020-11-12"\n},\n"1111013": {\n"lon": -90.109,\n"lat": 38.386,\n"date": "2017-05-03"\n},\n"554": {\n"lon": 151.171,\n"lat": -28.431,\n"date": "2021-11-30"\n}\n}'}

import os, sys, glob, time, json, hashlib, shutil, subprocess
from pathlib import Path
py = sys.executable
dataset = os.environ.get('PX1_AUDIT_DATASET', 'ks')
if dataset not in ('ks', 's1f', 'mm'):
    raise ValueError('PX1_AUDIT_DATASET must be ks, s1f or mm')
W = Path('/tmp/px1')
OUT = Path('/kaggle/working/tent_batch_audit') / dataset
OUT.mkdir(parents=True, exist_ok=True)
manifest = dict(dataset=dataset, method='tent_batch', status='started',
                schedule='event-wise 200 steps; Adam 1e-3; final per-chip batch statistics',
                files={k: hashlib.sha256(v.encode()).hexdigest() for k,v in FILES.items()})
def record():
    (OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2))
record()
bundle = glob.glob('/kaggle/input/**/run_kaggle.py', recursive=True)
if len(bundle) != 1:
    raise RuntimeError(f'Expected exactly one PX1 code bundle, found {len(bundle)}')
sys.path.insert(0, str(Path(bundle[0]).parent))
import run_kaggle
processes = []
handles = []
try:
    run_kaggle.setup('mmflood') if dataset == 'mm' else run_kaggle.setup()
    for name, content in FILES.items():
        (W / name).write_text(content)
    for name in ('r34_patch.py', 'rev_patch.py'):
        subprocess.run([py, name, '.'], cwd=W, check=True)
    jobs = []
    if dataset == 'ks':
        for seed in range(3):
            for part in range(2):
                jobs.append((f'ks_s{seed}_p{part}', part, [py, '-u', 'ks_eval.py', '--data', 'data',
                    '--ks', 'data_ks', '--runs', 'runs', '--model-seed', str(seed), '--out', f'runs_ks_tentbatch_p{part}',
                    '--methods', 'tent_batch', '--variants', 'V0', '--part', str(part), '--nparts', '2']))
    elif dataset == 'mm':
        for seed in range(3):
            for part in range(2):
                jobs.append((f'mm_s{seed}_p{part}', part, [py, '-u', 'mm_eval.py', '--data', 'data',
                    '--mm', 'data_mm', '--runs', 'runs', '--model-seed', str(seed), '--out', 'runs_mm_tentbatch',
                    '--methods', 'tent_batch', '--part', str(part), '--nparts', '2']))
    else:
        jobs.append(('s1f', 0, [py, '-u', 'tta_px1.py', '--data', 'data', '--runs', 'runs_loeo',
                              '--methods', 'tent_batch', '--model-seed', '0']))
    # At most one process per GPU. A failed job fails the notebook; preserve partial histograms.
    for offset in range(0, len(jobs), 2):
        processes = []
        handles = []
        for name, gpu, command in jobs[offset:offset+2]:
            handle = open(OUT / (name + '.log'), 'w')
            handles.append(handle)
            processes.append((name, subprocess.Popen(command, cwd=W,
                env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)), stdout=handle, stderr=subprocess.STDOUT)))
        failed = [(name, proc.wait()) for name, proc in processes]
        for handle in handles:
            handle.close()
        if any(code != 0 for _, code in failed):
            raise RuntimeError(f'Failed diagnostic jobs: {failed}')
        print('Completed:', failed, flush=True)
    manifest['status'] = 'completed'
except BaseException as exc:
    manifest['status'] = 'failed'
    manifest['error'] = repr(exc)
    raise
finally:
    for _, proc in processes:
        if proc.poll() is None:
            proc.terminate()
            proc.wait()
    for handle in handles:
        handle.close()
    patterns = {'ks': 'runs_ks_tentbatch_p*', 'mm': 'runs_mm_tentbatch', 's1f': 'runs_loeo'}
    for directory in W.glob(patterns[dataset]):
        for file in directory.glob('*.npz'):
            destination = OUT / directory.name
            destination.mkdir(exist_ok=True)
            shutil.copy2(file, destination / file.name)
    record()
print('TENT_BATCH_AUDIT_DONE', flush=True)
